# 03e — Kvantilregresjon

Estimerer prismodellen ved Q50 (median) og Q90 (90-persentil) for å besvare volatilitetsdelen av forskningsspørsmålet. Forskjellen $\beta_1^{(0,9)} - \beta_1^{(0,5)}$ måler om Stargate øker halefordelingen mer enn medianen.

**Input:** `intermediate/df_iso.parquet`

**Output:** `intermediate/preds_quantreg.parquet`, `intermediate/models_quantreg.pkl`

In [1]:
import pandas as pd
import statsmodels.api as sm

from src.config import (
    INTERMEDIATE_DIR, TARGET, QUANTREG_FEATURES, QUANTILES,
    TRAIN_YEARS, TEST_YEARS, apply_style,
)
from src.evaluation import eval_metrics
from src.model_training import (
    load_prepared_iso_data, split_features_target, make_prediction_frame,
    save_model_artifacts, fit_quantreg,
)

apply_style()

In [2]:
df_iso = load_prepared_iso_data(INTERMEDIATE_DIR)
X_train, y_train, X_test, y_test, train_mask, test_mask = split_features_target(
    df_iso,
    feature_cols=QUANTREG_FEATURES,
    target=TARGET,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
)

print(f"Trening: {len(X_train):,} ISO-timer")
print(f"Test:    {len(X_test):,} ISO-timer")
print(f"Kvantiler: {QUANTILES}")

Trening: 25,072 ISO-timer
Test:    9,844 ISO-timer
Kvantiler: [0.5, 0.9]


In [3]:
models = {}
coef_rows = []
for q in QUANTILES:
    model, _, _, _ = fit_quantreg(
        df=df_iso.loc[train_mask],
        y_col=TARGET,
        feature_cols=QUANTREG_FEATURES,
        quantile=q,
    )
    models[q] = model
    coef_rows.append({
        "Kvantil": q,
        "beta_cons_NO4": model.params["cons_NO4"],
        "std err": model.bse["cons_NO4"],
        "t-verdi": model.tvalues["cons_NO4"],
        "p-verdi": model.pvalues["cons_NO4"],
    })

coef_df = pd.DataFrame(coef_rows)
display(coef_df.round(4))

diff = models[0.9].params["cons_NO4"] - models[0.5].params["cons_NO4"]
print(f"\nVolatilitetseffekt (β₁^Q90 − β₁^Q50): {diff:.4f} NOK/MWh per MW")

,Kvantil,beta_cons_NO4,std err,t-verdi,p-verdi
0,0.5,0.1112,0.0049,22.8456,0.0
1,0.9,0.2214,0.0090,24.6891,0.0



Volatilitetseffekt (β₁^Q90 − β₁^Q50): 0.1102 NOK/MWh per MW


In [4]:
X_test_c = sm.add_constant(X_test, has_constant="add")

preds = make_prediction_frame(
    df=df_iso,
    mask=test_mask,
    actual=y_test,
    prediction_col="Q50",
    predictions=models[0.5].predict(X_test_c),
)
preds["Q90"] = models[0.9].predict(X_test_c).values

metrics = {
    "Q50": eval_metrics(preds["actual"], preds["Q50"]),
    "Q90": eval_metrics(preds["actual"], preds["Q90"]),
}
display(pd.DataFrame(metrics).T)

,MAE,RMSE,R²,N
Q50,125.4,207.5,0.3429,9844.0
Q90,293.0,360.5,-0.9837,9844.0


In [5]:
payload = {
    "model_name": "QuantReg",
    "prediction_cols": ["Q50", "Q90"],
    "feature_cols": QUANTREG_FEATURES,
    "target_col": TARGET,
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "quantiles": QUANTILES,
    "estimators": models,
    "metrics": metrics,
    "coefficients": coef_df,
    "volatility_effect": float(diff),
}

save_model_artifacts("quantreg", preds, payload, intermediate_dir=INTERMEDIATE_DIR)
print(f"Lagret {INTERMEDIATE_DIR}preds_quantreg.parquet")
print(f"Lagret {INTERMEDIATE_DIR}models_quantreg.pkl")

Lagret intermediate/preds_quantreg.parquet
Lagret intermediate/models_quantreg.pkl
